# Cross-validation sweep — BanglaPoliticalStance

Runs every model in `configs/` under **one** protocol: grouped stratified 5-fold CV, augmentation applied inside each training fold, out-of-fold predictions pooled.

**Setup:** Use a GPU runtime (**Runtime → Change runtime type → T4 GPU**).

**Dataset:** Public on HF at `kishormorol/BanglaPoliticalStance`. The annotated split (198 items) is used for evaluation. The repo code handles loading.

Each model writes `experiments/cv-<name>-<timestamp>/` with `cv.json` and `predictions.csv`. Download `experiments/` at the end — `bmpb leaderboard` builds the results table from it.

In [ ]:
!nvidia-smi -L
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Clone the repo

The repo is on GitHub. Add a secret named `GH_TOKEN` (key icon in the left sidebar) with a GitHub personal access token that has `repo` scope. If the repo is public, you can skip the token and clone directly.

In [ ]:
import subprocess, os

# Try with token first, fall back to public clone
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
    url = f"https://{token}@github.com/kishormorol/bangla-multimodal-political-stance.git"
except Exception:
    url = "https://github.com/kishormorol/bangla-multimodal-political-stance.git"

if not os.path.exists("bangla-multimodal-political-stance"):
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
%cd bangla-multimodal-political-stance
!git log --oneline -1

In [ ]:
!pip install -q -e . 2>&1 | tail -2
!pip install -q gdown 2>&1 | tail -1

## 2. The data

Two options:
- **Option A:** `bmpb data` mirrors the Drive folder (may hit rate limits — re-run to resume)
- **Option B:** Load directly from the public HF dataset `kishormorol/BanglaPoliticalStance`

We use Option A for the full pipeline, since `bmpb ingest` expects the raw Drive files.

In [ ]:
!python -m bmpb.cli data

### Article bodies

`bmpb backfill` refetches each item's source URL and pulls the article body out
of the page, so the text models see articles rather than 8-word headlines. It
recovers roughly two thirds of the corpus; the misses are outlets that refuse
scripted requests. `ingest` records `text_level` per item so the two populations
can be reported separately.

In [ ]:
!python -m bmpb.cli backfill --delay 1.0

In [ ]:
!python -m bmpb.cli ingest
!python -m bmpb.cli audit

## 3. The sweep

Text encoders are quick (~2 min each); vision-language models take longer (BLIP, ViLT at 384px). Run text first so you have early numbers.

**Models evaluated:**
- **Text:** majority, tfidf_logreg, BanglaBERT, mBERT, XLM-RoBERTa, BanglaELECTRA, mT5
- **Multimodal:** CLIP, ALIGN, BLIP, ViLT, FLAVA, CountVec+ViT

In [ ]:
!bash scripts/run_cv.sh configs/text

In [ ]:
!bash scripts/run_cv.sh configs/multimodal

## 4. Results

`leaderboard` puts the cross-validated rows in one table and keeps the originally
published numbers separate, since those came from three different protocols and
are not comparable with these.

In [ ]:
!python -m bmpb.cli leaderboard
print(open("reports/tables/leaderboard.md").read())

## 5. Take the runs home

`experiments/` and `reports/` are what the paper's tables are built from. Commit
them back, or download the archive and unpack it into the local checkout.

In [ ]:
!tar czf cv-runs.tar.gz experiments reports
from google.colab import files
files.download("cv-runs.tar.gz")